In [1]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings,ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
load_dotenv()

True

In [2]:
def get_transcript(video_id):
    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.list(video_id)
    print("Available transcripts:")

    for t in transcript_list:
        print(t.language_code, "-", t.language)

    try:
        transcript = transcript_list.find_transcript(
            ['en', 'en-US', 'hi']
        ).fetch()

    except:
        transcript = next(iter(transcript_list)).fetch()

    return transcript


video_id = "a3C1DMswClQ"
transcript = get_transcript(video_id)
full_text = " ".join([x.text for x in transcript])
print(full_text[:1000])

Available transcripts:
en - English (auto-generated)
backend is huge and if we start discussing every single component that could be part of it we will be stuck here for years so what we will do is we will only discuss those topics which are used in majority of the code bases let's say 90% of them with that in mind let's talk about HTTP protocol the medium through which our browsers talk to our servers either to send data or to receive data from it and as I said there are a lot of other ways and protocols which clients and servers use to communicate with each other and HTTP being one of the most used ones we will focus on that now there are two ideas which are at the heart of HTTP protocol the first one being tessness what does it mean tessness basically means it has no memory of past interactions so each HTTP request carries all the necessary information for the server to process it such as headers or URLs and methods which we will see in a bit and after the server responds it forgets

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([full_text])
len(chunks)

85

In [6]:
embeddings = HuggingFaceEndpointEmbeddings(
    repo_id="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = FAISS.from_documents(chunks, embeddings)

In [7]:
vector_store.index_to_docstore_id

{0: '64ddfd0e-0148-4176-8657-60e876a6c31b',
 1: '223fc1b3-c056-45c6-a1fd-a54046e2efdb',
 2: '0bd204ed-7813-4c10-b6f0-d1d7ab367ef4',
 3: '8a10aa4d-0387-4f84-8e0f-f32f36ed5c5e',
 4: 'beb2a864-cc4d-4206-8137-a510908f3b2b',
 5: '6233f9da-0d09-4085-94ff-d054e39b5888',
 6: '6724ff97-f004-4403-8ef5-01526bc77e27',
 7: '5816d834-5cfe-41c5-8611-52259ed08d69',
 8: '4632e362-c25f-46ac-a075-e771bae0256b',
 9: '14ce1fd9-eb15-4690-9e54-b2f668a0118b',
 10: 'ceb96c63-fd9a-4406-b0ed-3a942c8a8481',
 11: '40e69ad1-4c55-4cae-91d8-b2c802169a96',
 12: '6c534638-a007-444f-8b2e-45bd42fbc72a',
 13: 'a64c239f-d103-4469-997c-ca1eb2869bc1',
 14: 'a7b5a485-5700-4063-8857-67ffc6346a97',
 15: '6bf02ba5-7052-42c5-a733-f15f35adbf53',
 16: '0a4af59b-4f04-46ca-b93f-a5995f944dcd',
 17: '80983bac-2857-4c85-abb4-f7cd9f31c9af',
 18: '88befba1-076a-4053-a5a9-eff6cac6bd08',
 19: '6b81ed93-19c0-4a02-972e-96599d869efc',
 20: '6aafd87f-68b7-44a5-b585-fed1620de604',
 21: 'da2c2532-6b0f-4097-9d0a-95803e6be55a',
 22: '13e5dad3-58ed-

In [11]:
vector_store.get_by_ids(['3a497b55-be56-408d-91df-fb41d0334ebd'])

[Document(id='3a497b55-be56-408d-91df-fb41d0334ebd', metadata={}, page_content="and overally this is all you need to know about HTTP at least of course there are more stuff to read if you want on HTTP or TLS or PCP protocol different different components of HTTP but in order to work on backend systems if you understand this much if you internalize this much and you can visualize the whole flow of all the the components that we talked about today then you're good to go you'll be able to understand all you need to understand and you'll be able to debug most of the stuff now that you understand how the system works behind the scenes and what are the components that come into play in different different flows that's all about http")]

In [10]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [12]:
retriever.invoke('what is https')

[Document(id='beb2a864-cc4d-4206-8137-a510908f3b2b', metadata={}, page_content='by the server and throughout our discussion we can go ahead and safely assume that HTTP and https are interchangeable because https to oversimplify it just a more secure version of HTTP but the underlying principles are the same with more security features like encryption and security certificates or TLS stuff like that which boarders too much into the network engineering domain also to send some kind of request or receive some kind of response first the client and the server need to establish some kind of connection mechanism right otherwise how are they going to communicate what is the medium of the communication and for that sgtp uses TCP TCP which is a protocol which is a transmission protocol essentially HTTP does not require the underlying transport protocol to be connection based it only requires it to be reliable and not lose message messages at minimum presenting an error in such cases and among th

In [13]:
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V3.1-Terminus",
    task="conversational",
)
model = ChatHuggingFace(llm=llm)

In [14]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [15]:
question          = "what is cors"
retrieved_docs    = retriever.invoke(question)
retrieved_docs

[Document(id='6e488d94-8701-43c8-8304-0ab8bdef207d', metadata={}, page_content="cross origin request there are primarily two types of flows one is a simple request flow and another one is a pre-f flighter request flow first let's look at the simple request imagine our client our frontend is at the Domain example.com and our server is at the Domain api. example.com and we make a get request which looks something like like this this is how the flow looks like the client sends a request the browser automatically adds the origin header to indicate the origin of the request the request uses a simple method which are usually get post or head and then it reaches the server the server checks for the origin header against it C policy if the origin is allowed the server includes the access control allow origin header in the response and it sends the response response the server responds with the resource and includes the necessary course headers for example Access Control allow origin header and

In [16]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"cross origin request there are primarily two types of flows one is a simple request flow and another one is a pre-f flighter request flow first let's look at the simple request imagine our client our frontend is at the Domain example.com and our server is at the Domain api. example.com and we make a get request which looks something like like this this is how the flow looks like the client sends a request the browser automatically adds the origin header to indicate the origin of the request the request uses a simple method which are usually get post or head and then it reaches the server the server checks for the origin header against it C policy if the origin is allowed the server includes the access control allow origin header in the response and it sends the response response the server responds with the resource and includes the necessary course headers for example Access Control allow origin header and this is what the response looks like it has the appropriate course header\n\ns

In [17]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      cross origin request there are primarily two types of flows one is a simple request flow and another one is a pre-f flighter request flow first let's look at the simple request imagine our client our frontend is at the Domain example.com and our server is at the Domain api. example.com and we make a get request which looks something like like this this is how the flow looks like the client sends a request the browser automatically adds the origin header to indicate the origin of the request the request uses a simple method which are usually get post or head and then it reaches the server the server checks for the origin header against it C policy if the origin is allowed the server includes the access control allow origin header in the response and it sends the response response the server responds

In [20]:
answer = model.invoke(final_prompt)
print(answer.content)

Based on the provided transcript context, CORS (Cross-Origin Resource Sharing) is a mechanism that involves the browser and server to handle cross-origin requests. It uses two primary flows: simple requests and pre-flighted requests. The server must include appropriate CORS headers (like `Access-Control-Allow-Origin`) in its response to allow the browser to accept the response from a different origin, otherwise the browser blocks it.


In [21]:
print(model.invoke(prompt.invoke({"context":context_text, "question":"what is the ipl format"})))

content="I don't know." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 1021, 'total_tokens': 1027}, 'model_name': 'deepseek-ai/DeepSeek-V3.1-Terminus', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--9355c636-a956-4b9d-b475-aad61c3ce0da-0' usage_metadata={'input_tokens': 1021, 'output_tokens': 6, 'total_tokens': 1027}


# LLM chain

In [22]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [23]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [26]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})
parser = StrOutputParser()
main_chain = parallel_chain | prompt | model | parser

In [27]:
main_chain.invoke('what are the topics discussed in the video')

'Based on the provided transcript, the topics discussed in the video are:\n\n1. **Content Negotiation**: This includes media type, language negotiation, and encoding negotiation, with a mention of HTTP compression as part of this topic.\n2. **Handling Large Requests and Responses**: Specifically, using multipart requests for sending large files from client to server, and using chunk transfer or text event stream for transferring large files from server to client.\n3. **SSL/TLS/HTTPS**: A brief explanation of what these terms mean for securing communications.'

In [28]:
main_chain.invoke("who is homelander")

"I don't know."